In [1]:
!git clone https://github.com/syvlo/RSVQA.git

Cloning into 'RSVQA'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 36 (delta 6), reused 27 (delta 2), pack-reused 0
Unpacking objects: 100% (36/36), 38.04 KiB | 2.93 MiB/s, done.


In [2]:
!git clone https://github.com/syvlo/RSVQAxBEN.git

Cloning into 'RSVQAxBEN'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 3), reused 13 (delta 2), pack-reused 0
Unpacking objects: 100% (17/17), 1.47 MiB | 9.18 MiB/s, done.


In [3]:
# from gensim.models import KeyedVectors

# # Load Word2Vec model
# word2vec_model_path = "/kaggle/input/wvvector/GoogleNews-vectors-negative300.bin"
# word2vec_model = KeyedVectors.load_word2vec_format(word2vec_model_path, binary=True)

In [4]:
!pip install -r "/kaggle/working/RSVQA/requirements.txt"

In [5]:
#!wget https://zenodo.org/api/records/6344334/files-archive

In [6]:
#!unzip /kaggle/working/files-archive

In [7]:
#!unzip /kaggle/working/Images_LR.zip

In [8]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
@author: sylvain
"""

# Classe définissant le jeu de donnée VQA au format pytorch

import os.path
import json
import random

import numpy as np
from skimage import io

from torch.utils.data import Dataset
import torchvision.transforms as T

RANDOM_SEED = 42


class VQALoader(Dataset):
    def __init__(self, imgFolder, images_file, questions_file, answers_file, encoder_questions, encoder_answers, train=True, ratio_images_to_use = 1, transform=None, patch_size=512):
        self.transform = transform
        self.encoder_questions = encoder_questions
        self.encoder_answers = encoder_answers
        self.train = train
        
        
        vocab = self.encoder_questions.words
        self.relationalWords = [vocab['top'], vocab['bottom'], vocab['right'], vocab['left']]
        
        with open(questions_file) as json_data:
            self.questionsJSON = json.load(json_data)
            
        with open(answers_file) as json_data:
            self.answersJSON = json.load(json_data)
            
        with open(images_file) as json_data:
            self.imagesJSON = json.load(json_data)
        
        images = [img['id'] for img in self.imagesJSON['images'] if img['active']]
        images = images[:int(len(images)*ratio_images_to_use)]
        self.images = np.empty((len(images), patch_size, patch_size, 3))
        
        self.len = 0
        for image in images:
            self.len += len(self.imagesJSON['images'][image]['questions_ids'])
        self.images_questions_answers = [[None] * 4] * self.len
        
        index = 0
        for i, image in enumerate(images):
            img = io.imread(os.path.join(imgFolder, str(image)+'.tif'))
            self.images[i, :, :, :] = img
            for questionid in self.imagesJSON['images'][image]['questions_ids']:
                question = self.questionsJSON['questions'][questionid]
            
                question_str = question["question"]
                type_str = question["type"]
                answer_str = self.answersJSON['answers'][question["answers_ids"][0]]['answer']
            
                self.images_questions_answers[index] = [self.encoder_questions.encode(question_str), self.encoder_answers.encode(answer_str), i, type_str]
                index += 1
    def __len__(self):
        return self.len
    
    def __getitem__(self, idx):
        question = self.images_questions_answers[idx]
        img = self.images[question[2],:,:,:]
        if self.train and not self.relationalWords[0] in question[0] and not self.relationalWords[1] in question[0] and not self.relationalWords[2] in question[0] and not self.relationalWords[3] in question[0]:
            if random.random() < .5:
                img = np.flip(img, axis = 0)
            if random.random() < .5:
                img = np.flip(img, axis = 1)
            if random.random() < .5:
                img = np.rot90(img, k=1)
            if random.random() < .5:
                img = np.rot90(img, k=3)
        if self.transform:
            imgT = self.transform(img.copy())
        if self.train:
            return np.array(question[0], dtype='int16'), np.array(question[1], dtype='int16'), imgT, question[3]
        else:
            return np.array(question[0], dtype='int16'), np.array(question[1], dtype='int16'), imgT, question[3], T.ToTensor()(img / 255)   

In [9]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.nn.functional as F

#from models.skipthoughts import skipthoughts
# From https://github.com/Cadene/vqa.pytorch


def process_lengths(input):
    max_length = input.size(1)
    lengths = list(max_length - input.data.eq(0).sum(1).squeeze())
    return lengths

def select_last(x, lengths):
    batch_size = x.size(0)
    seq_length = x.size(1)
    mask = x.data.new().resize_as_(x.data).fill_(0)
    for i in range(batch_size):
        mask[i][lengths[i]-1].fill_(1)
    mask = Variable(mask)
    x = x.mul(mask)
    x = x.sum(1).view(batch_size, x.size(2))
    return x

class LSTM(nn.Module):

    def __init__(self, vocab, emb_size, hidden_size, num_layers):
        super(LSTM, self).__init__()
        self.vocab = vocab
        self.emb_size = emb_size
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding = nn.Embedding(num_embeddings=len(self.vocab)+1,
                                      embedding_dim=emb_size,
                                      padding_idx=0)
        self.rnn = nn.LSTM(input_size=emb_size, hidden_size=hidden_size, num_layers=num_layers)

    def forward(self, input):
        lengths = process_lengths(input)
        x = self.embedding(input) # seq2seq
        output, hn = self.rnn(x)
        output = select_last(output, lengths)
        return output


class TwoLSTM(nn.Module):

    def __init__(self, vocab, emb_size, hidden_size):
        super(TwoLSTM, self).__init__()
        self.vocab = vocab
        self.emb_size = emb_size
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(num_embeddings=len(self.vocab)+1,
                                      embedding_dim=emb_size,
                                      padding_idx=0)
        self.rnn_0 = nn.LSTM(input_size=emb_size, hidden_size=hidden_size, num_layers=1)
        self.rnn_1 = nn.LSTM(input_size=hidden_size, hidden_size=hidden_size, num_layers=1)

    def forward(self, input):
        lengths = process_lengths(input)
        x = self.embedding(input) # seq2seq
        x = getattr(F, 'tanh')(x)
        x_0, hn = self.rnn_0(x)
        vec_0 = select_last(x_0, lengths)

        # x_1 = F.dropout(x_0, p=0.3, training=self.training)
        # print(x_1.size())
        x_1, hn = self.rnn_1(x_0)
        vec_1 = select_last(x_1, lengths)
        
        vec_0 = F.dropout(vec_0, p=0.3, training=self.training)
        vec_1 = F.dropout(vec_1, p=0.3, training=self.training)
        output = torch.cat((vec_0, vec_1), 1)
        return output
        

def factory(vocab_words, opt):
    if opt['arch'] == 'skipthoughts':
        st_class = getattr(skipthoughts, opt['type'])
        seq2vec = st_class(opt['dir_st'],
                           vocab_words,
                           dropout=opt['dropout'],
                           fixed_emb=opt['fixed_emb'])
    elif opt['arch'] == '2-lstm':
        seq2vec = TwoLSTM(vocab_words,
                          opt['emb_size'],
                          opt['hidden_size'])
    elif opt['arch'] == 'lstm':
        seq2vec = TwoLSTM(vocab_words,
                          opt['emb_size'],
                          opt['hidden_size'],
                          opt['num_layers'])
    elif opt['arch'] == 'bert':
        seq2vec = BertEncoder(opt['bert_model'], opt['hidden_size'], opt['dropout'])
        
    elif opt['arch'] == 'word2vec':  # New condition for Word2Vec
        # Load your pre-trained model (loaded in the previous code snippet)
        word2vec_model = word2vec_model  
        # Create a simple embedding lookup mechanism
        def seq2vec(sentences):
            embeddings = []
            for sentence in sentences:
                word_embeddings = [word2vec_model[word] for word in sentence if word in word2vec_model]
                
                # Handle sentence representation (you'll need a strategy here)
                if word_embeddings:
                    sentence_embedding = np.mean(word_embeddings, axis=0) 
                else:
                    sentence_embedding = np.zeros(300)  # Default for OOV cases

                embeddings.append(sentence_embedding)
            return embeddings

    else:
        raise NotImplementedError
    return seq2vec


if __name__ == '__main__':

    vocab = ['robots', 'are', 'very', 'cool', '<eos>', 'BiDiBu']
    lstm = TwoLSTM(vocab, 300, 1024).cuda()

    myinput = Variable(torch.LongTensor([
        [1,2,3,4,5,0,0],
        [6,1,2,3,3,4,5],
        [6,1,2,3,3,4,5]
    ])).cuda()
    output = lstm(myinput)
    print(output.shape)

torch.Size([3, 2048])


In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
@author: sylvain
"""

# Encodeur du vocabulaireß

import json

MAX_ANSWERS = 100
LEN_QUESTION = 20


class VocabEncoder():
    # Création du dictionnaire en parcourant l'ensemble du JSON (des questions ou des réponses)
    def __init__(self, JSONFile, string=None, questions=True, range_numbers=False):
        self.encoder_type = 'answer'
        if questions:
            self.encoder_type = 'question'
        self.questions = questions
        self.range_numbers = range_numbers
        
        words = {}  
        
        if JSONFile != None:
            with open(JSONFile) as json_data:
                self.data = json.load(json_data)[self.encoder_type + 's']
        else:
            if questions:
                self.data = [{'question':string}]
            else:
                self.data = [{'answer':string}]
            
        
        for i in range(len(self.data)):
            if self.data[i]["active"]:
                sentence = self.data[i][self.encoder_type]
                if sentence[-1] == "?" or sentence[-1] == ".":
                    sentence = sentence[:-1]
                
                tokens = sentence.split()
                for token in tokens:
                    token = token.lower()
                    if range_numbers and token.isdigit() and not questions:
                        num = int(token)
                        if num > 0 and num <= 10:
                            token = "between 0 and 10"
                        if num > 10 and num <= 100:
                            token = "between 10 and 100"
                        if num > 100 and num <= 1000:
                            token = "between 100 and 1000"
                        if num > 1000:
                            token = "more than 1000"

                    if token[-2:] == 'm2' and not questions:
                        num = int(token[:-2])
                        if num > 0 and num <= 10:
                            token = "between 0m2 and 10m2"
                        if num > 10 and num <= 100:
                            token = "between 10m2 and 100m2"
                        if num > 100 and num <= 1000:
                            token = "between 100m2 and 1000m2"
                        if num > 1000:
                            token = "more than 1000m2"
                    if token not in words:
                        words[token] = 1
                    else:
                        words[token] += 1
                
        sorted_words = sorted(words.items(), key=lambda kv: kv[1], reverse=True)
        self.words = {'<EOS>':0}
        self.list_words = ['<EOS>']
        for i, word in enumerate(sorted_words):
            if self.encoder_type == 'answer':
                if i >= MAX_ANSWERS:
                    break
            self.words[word[0]] = i + 1
            self.list_words.append(word[0])
    
    #Encodage d'une phrase (question ou réponse) à partir du dictionnaire crée plus tôt.        
    def encode(self, sentence):
        res = []
        if sentence[-1] == "?" or sentence[-1] == ".":
            sentence = sentence[:-1]
            
        tokens = sentence.split()
        for token in tokens:
            token = token.lower()
            if self.range_numbers and token.isdigit() and not self.questions:
                num = int(token)
                if num > 0 and num <= 10:
                    token = "between 0 and 10"
                if num > 10 and num <= 100:
                    token = "between 10 and 100"
                if num > 100 and num <= 1000:
                    token = "between 100 and 1000"
                if num > 1000:
                    token = "more than 1000"
                    
            if token[-2:] == 'm2' and not self.questions:
                num = int(token[:-2])
                if num > 0 and num <= 10:
                    token = "between 0m2 and 10m2"
                if num > 10 and num <= 100:
                    token = "between 10m2 and 100m2"
                if num > 100 and num <= 1000:
                    token = "between 100m2 and 1000m2"
                if num > 1000:
                    token = "more than 1000m2"
            res.append(self.words[token])
        
        if self.questions:
            res.append(self.words['<EOS>'])
        
        if self.questions:
            while len(res) < LEN_QUESTION:
                res.append(self.words['<EOS>'])
            res = res[:LEN_QUESTION]
        return res
    
    
    def getVocab(self):
        return self.list_words
    
    #Décodage d'une phrase (seulement utilisé pour l'affichage des résultats)
    def decode(self, sentence):
        res = ""
        for i in sentence:
            if i == 0:
                break
            res += self.list_words[i]
            res += " "
        res = res[:-1]
        if self.questions:
            res += "?"
        return res
        
            
            
            
        

In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Fri Nov 30 09:55:48 2018

@author: sylvain
"""

'''from torchvision import models as torchmodels
import torch.nn as nn
#import models.seq2vec
import torch.nn.functional as F
import torch

VISUAL_OUT = 2048
QUESTION_OUT = 2400
FUSION_IN = 1200
FUSION_HIDDEN = 256
DROPOUT_V = 0.5
DROPOUT_Q = 0.5
DROPOUT_F = 0.5

class VQAModel(nn.Module):
    def __init__(self, vocab_questions, vocab_answers, input_size = 512):
        super(VQAModel, self).__init__()
        
        self.vocab_questions = vocab_questions
        self.vocab_answers = vocab_answers
        self.num_classes = len(self.vocab_answers)
        
        self.dropoutV = torch.nn.Dropout(DROPOUT_V)
        self.dropoutQ = torch.nn.Dropout(DROPOUT_Q)
        self.dropoutF = torch.nn.Dropout(DROPOUT_F)
        self.seq2vec = factory(self.vocab_questions, {'arch': '2-lstm', 'emb_size': 300,'hidden_size':1024 })
        for param in self.seq2vec.parameters():
            param.requires_grad = False
        self.linear_q = nn.Linear(QUESTION_OUT, FUSION_IN)
        
        self.visual = torchmodels.resnet152(pretrained=True)
        extracted_layers = list(self.visual.children())
        extracted_layers = extracted_layers[0:8] #Remove the last fc and avg pool
        self.visual = torch.nn.Sequential(*(list(extracted_layers)))
        for param in self.visual.parameters():
            param.requires_grad = False
        
        output_size = (input_size / 32)**2
        self.visual = torch.nn.Sequential(self.visual, torch.nn.Conv2d(2048,int(2048/output_size),1))
        self.linear_v = nn.Linear(VISUAL_OUT, FUSION_IN)
        
        self.linear_classif1 = nn.Linear(FUSION_IN, FUSION_HIDDEN)
        self.linear_classif2 = nn.Linear(FUSION_HIDDEN, self.num_classes)
        
    def forward(self, input_v, input_q):
        x_v = self.visual(input_v).view(-1, VISUAL_OUT)
        x_v = self.dropoutV(x_v)
        print(f"x_v: {x_v.shape} ")
        x_v = self.linear_v(x_v)
        x_v = nn.Tanh()(x_v)

        x_q = self.seq2vec(input_q)
        x_q = self.dropoutQ(x_q)  # Correct dropout
        print(f"x_q: {x_q.shape} ")
        x_q = self.linear_q(x_q)
        x_q = nn.Tanh()(x_q)

        print(f"x_v2: {x_v.shape} ")
        print(f"x_q2: {x_q.shape} ")
        # Perform element-wise multiplication
        x = torch.matmul(x_q, x_v.t())
        x = nn.Tanh()(x)
        x = self.dropoutF(x)

        # Proceed with classification layers
        x = self.linear_classif1(x)
        x = nn.Tanh()(x)
        x = self.dropoutF(x)
        x = self.linear_classif2(x)

        return x
        '''

'from torchvision import models as torchmodels\nimport torch.nn as nn\n#import models.seq2vec\nimport torch.nn.functional as F\nimport torch\n\nVISUAL_OUT = 2048\nQUESTION_OUT = 2400\nFUSION_IN = 1200\nFUSION_HIDDEN = 256\nDROPOUT_V = 0.5\nDROPOUT_Q = 0.5\nDROPOUT_F = 0.5\n\nclass VQAModel(nn.Module):\n    def __init__(self, vocab_questions, vocab_answers, input_size = 512):\n        super(VQAModel, self).__init__()\n        \n        self.vocab_questions = vocab_questions\n        self.vocab_answers = vocab_answers\n        self.num_classes = len(self.vocab_answers)\n        \n        self.dropoutV = torch.nn.Dropout(DROPOUT_V)\n        self.dropoutQ = torch.nn.Dropout(DROPOUT_Q)\n        self.dropoutF = torch.nn.Dropout(DROPOUT_F)\n        self.seq2vec = factory(self.vocab_questions, {\'arch\': \'2-lstm\', \'emb_size\': 300,\'hidden_size\':1024 })\n        for param in self.seq2vec.parameters():\n            param.requires_grad = False\n        self.linear_q = nn.Linear(QUESTION_OUT,

In [12]:
from transformers import BertModel, BertTokenizer

class BertEncoder(nn.Module):
    def __init__(self, bert_model='bert-base-uncased', hidden_size=768, dropout=0.1):
        super(BertEncoder, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model)
        self.dropout = nn.Dropout(dropout)
        self.hidden_size = hidden_size

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        pooled_output = outputs[1]
        pooled_output = self.dropout(pooled_output)
        return pooled_output

In [13]:
import torch
import torch.nn as nn
import torchvision.models as torchmodels

class BertEncoder(nn.Module):
    def __init__(self, bert_model='bert-base-uncased', hidden_size=768, dropout=0.1):
        super(BertEncoder, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model)
        self.dropout = nn.Dropout(dropout)
        self.hidden_size = hidden_size

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        pooled_output = outputs[1]
        pooled_output = self.dropout(pooled_output)
        return pooled_output

class BilinearAttention(nn.Module):
    def __init__(self, v_dim, q_dim, num_hid):
        super(BilinearAttention, self).__init__()
        self.v_proj = nn.Linear(v_dim, num_hid)
        self.q_proj = nn.Linear(q_dim, num_hid)
        self.dropout = nn.Dropout(0.5)
        self.weights = nn.Parameter(torch.rand(num_hid, num_hid))

    def forward(self, v, q):
        v_proj = self.dropout(self.v_proj(v))
        q_proj = self.dropout(self.q_proj(q))
        v_proj = v_proj.unsqueeze(2)
        q_proj = q_proj.unsqueeze(1)
        attn = torch.matmul(torch.matmul(q_proj, self.weights), v_proj)
        attn = attn.squeeze(1).squeeze(1)
        return attn

class VQAModel(nn.Module):
    def __init__(self, vocab_questions, vocab_answers, bert_model='bert-base-uncased', input_size=512):
        super(VQAModel, self).__init__()
        
        self.vocab_questions = vocab_questions
        self.vocab_answers = vocab_answers
        self.num_classes = len(self.vocab_answers)
        
        self.bert_tokenizer = BertTokenizer.from_pretrained(bert_model)
        self.bert_encoder = BertEncoder(bert_model)
        
        # Visual model (ResNet)
        self.visual = torchmodels.resnet152(pretrained=True)
        extracted_layers = list(self.visual.children())[0:8]  # Remove the last fc and avg pool
        self.visual = nn.Sequential(*extracted_layers)
        for param in self.visual.parameters():
            param.requires_grad = False
        
        # Linear layer for visual features
        VISUAL_OUT = 2048  # Update this according to the output size of ResNet
        self.linear_v = nn.Linear(VISUAL_OUT, input_size)
        
        # Bilinear Attention
        self.attention = BilinearAttention(v_dim=input_size, q_dim=self.bert_encoder.hidden_size, num_hid=512)
        
        # Classification layers
        self.linear_classif1 = nn.Linear(input_size, 512)
        self.linear_classif2 = nn.Linear(512, self.num_classes)
        
    def forward(self, input_v, input_q):
        # Visual feature extraction
        x_v = self.visual(input_v)
        x_v = nn.AvgPool2d(kernel_size=7)(x_v)  # Adjust kernel size based on input size
        x_v = torch.flatten(x_v, start_dim=1)
        x_v = self.linear_v(x_v)

        # Question feature extraction
        input_ids = self.bert_tokenizer(input_q, return_tensors='pt', padding=True, truncation=True).to(input_v.device)
        x_q = self.bert_encoder(input_ids=input_ids['input_ids'], attention_mask=input_ids['attention_mask'])

        # Bilinear Attention
        attn_scores = self.attention(x_v, x_q)
        attended_v = x_v * attn_scores.unsqueeze(-1)

        # Classification layers
        x = self.linear_classif1(attended_v)
        x = nn.ReLU()(x)
        x = self.linear_classif2(x)

        return x

In [14]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
@author: sylvain
"""

# Script principal d'apprentissage

import matplotlib
matplotlib.use('Agg')

#from models import model
#import VQALoader
#import VocabEncoder
import torchvision.transforms as T
import torch
from torch.autograd import Variable

import numpy as np
import matplotlib.pyplot as plt

import pickle
import os
import datetime
from shutil import copyfile


def train(model, train_dataset, validate_dataset, batch_size, num_epochs, learning_rate, modeltype, pre_trained_model=None, Dataset='HR'):
#     if pre_trained_model is not None:
#         model.load_state_dict(torch.load(pre_trained_model))
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    validate_loader = torch.utils.data.DataLoader(validate_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    criterion = torch.nn.CrossEntropyLoss()
        
    trainLoss = []
    valLoss = []
    if Dataset == 'HR':
        accPerQuestionType = {'area': [], 'presence': [], 'count': [], 'comp': []}
    else:
        accPerQuestionType = {'rural_urban': [], 'presence': [], 'count': [], 'comp': []}
    OA = []
    AA = []
    for epoch in range(num_epochs):
        model.train()
        runningLoss = 0
        for i, data in enumerate(train_loader, 0):
            if i % 1000 == 999:
                print(i/len(train_loader))
            question, answer, image, _ = data
            question_str = [encoder_questions.decode(q.cpu().numpy()) for q in question]
            answer = Variable(answer.long()).cuda().resize_(len(question_str))
            image = Variable(image.float()).cuda()
            
            pred = model(image, question_str)
            loss = criterion(pred, answer)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            runningLoss += loss.cpu().item() * len(question_str)
            
        trainLoss.append(runningLoss / len(train_dataset))
        print('epoch %d train_loss:--------------------- %.3f' % (epoch, trainLoss[epoch]))
        
        with torch.no_grad():
            model.eval()
            runningLoss = 0
            if Dataset == 'HR':
                countQuestionType = {'presence': 0, 'count': 0, 'comp': 0, 'area': 0}
                rightAnswerByQuestionType = {'presence': 0, 'count': 0, 'comp': 0, 'area': 0}
            else:
                countQuestionType = {'presence': 0, 'count': 0, 'comp': 0, 'rural_urban': 0}
                rightAnswerByQuestionType = {'presence': 0, 'count': 0, 'comp': 0, 'rural_urban': 0}
            count_q = 0
            for i, data in enumerate(validate_loader, 0):
                if i % 1000 == 999:
                    print(i/len(validate_loader))
                question, answer, image, type_str, image_original = data
                question_str = [encoder_questions.decode(q.cpu().numpy()) for q in question]
                answer = Variable(answer.long()).cuda().resize_(len(question_str))
                image = Variable(image.float()).cuda()
                
                pred = model(image, question_str)
                loss = criterion(pred, answer)
                runningLoss += loss.cpu().item() * len(question_str)
                
                answer = answer.cpu().numpy()
                pred = np.argmax(pred.cpu().detach().numpy(), axis=1)
                for j in range(len(question_str)):
                    countQuestionType[type_str[j]] += 1
                    if answer[j] == pred[j]:
                        rightAnswerByQuestionType[type_str[j]] += 1

                if i % 50 == 2 and i < 999:
                    fig1, f1_axes = plt.subplots(ncols=1, nrows=2)
                    viz_img = T.ToPILImage()(image_original[0].float().data.cpu())
                    viz_question = question_str[0]
                    viz_answer = encoder_answers.decode([answer[0]])
                    viz_pred = encoder_answers.decode([pred[0]])
    
                    f1_axes[0].imshow(viz_img)
                    f1_axes[0].axis('off')
                    f1_axes[0].set_title(viz_question)
                    f1_axes[1].axis('off')
                    f1_axes[1].set_title(viz_answer)
                    text = f1_axes[1].text(0.5, -0.1, viz_pred, size=12, horizontalalignment='center',
                                              verticalalignment='center', transform=f1_axes[1].transAxes)
                    plt.savefig('/tmp/VQA.png')
                    plt.close(fig1)
                        
            valLoss.append(runningLoss / len(validate_dataset))
            print('epoch %d val loss------------------ %.3f' % (epoch, valLoss[epoch]))
        
            numQuestions = 0
            numRightQuestions = 0
            currentAA = 0
            for type_str in countQuestionType.keys():
                if countQuestionType[type_str] > 0:
                    accPerQuestionType[type_str].append(rightAnswerByQuestionType[type_str] * 1.0 / countQuestionType[type_str])
                numQuestions += countQuestionType[type_str]
                numRightQuestions += rightAnswerByQuestionType[type_str]
                currentAA += accPerQuestionType[type_str][epoch]
                
            OA.append(numRightQuestions * 1.0 / numQuestions)
            AA.append(currentAA * 1.0 / 4)
        
        torch.save(model.state_dict(), "ModelRSVQA.pth")

    return model


if __name__ == '__main__':
    disable_log = True
    batch_size = 70
    num_epochs = 1
    learning_rate = 0.00001
    ratio_images_to_use = 1
    modeltype = 'Simple'
    Dataset = 'LR'
    pre_trained_model_path = "/kaggle/input/bertattmatrix/BertAttention.pth"

    if Dataset == 'LR':
        data_path = '/kaggle/input/vqars1/6344334/'#'/raid/home/sylvain/RSVQA_USGS_data/'#'../AutomaticDB/'
        allquestionsJSON = os.path.join(data_path, 'all_questions.json')
        allanswersJSON = os.path.join(data_path, 'all_answers.json')
        questionsJSON = os.path.join(data_path, 'LR_split_train_questions.json')
        answersJSON = os.path.join(data_path, 'LR_split_train_answers.json')
        imagesJSON = os.path.join(data_path, 'LR_split_train_images.json')
        questionsvalJSON = os.path.join(data_path, 'LR_split_val_questions.json')
        answersvalJSON = os.path.join(data_path, 'LR_split_val_answers.json')
        imagesvalJSON = os.path.join(data_path, 'LR_split_val_images.json')
        images_path = os.path.join(data_path, 'Images_LR/')
    else:
        data_path = '/raid/home/sylvain/RSVQA_USGS_data/'
        images_path = os.path.join(data_path, 'dataUSGS/')
    encoder_questions = VocabEncoder(allquestionsJSON, questions=True)
    if Dataset == "LR":
        encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = True)
    else:
        encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = False)

    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]
    transform = T.Compose([
        T.ToTensor(),            
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
      ])
    
    if Dataset == 'LR':
        patch_size = 256
    else:
        patch_size = 512   
    train_dataset = VQALoader(images_path, imagesJSON, questionsJSON, answersJSON, encoder_questions, encoder_answers, train=True, ratio_images_to_use=ratio_images_to_use, transform=transform, patch_size = patch_size)
    validate_dataset = VQALoader(images_path, imagesvalJSON, questionsvalJSON, answersvalJSON, encoder_questions, encoder_answers, train=False, ratio_images_to_use=ratio_images_to_use, transform=transform, patch_size = patch_size)
    
    
    RSVQA = VQAModel(encoder_questions.getVocab(), encoder_answers.getVocab(), input_size = patch_size).cuda()
#     if os.path.exists(pre_trained_model_path):
#         RSVQA.load_state_dict(torch.load(pre_trained_model_path))
    RSVQA = train(RSVQA, train_dataset, validate_dataset, batch_size, num_epochs, learning_rate, modeltype,pre_trained_model_path, Dataset)
    
    

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet152-394f9c45.pth" to /root/.cache/torch/hub/checkpoints/resnet152-394f9c45.pth
100%|██████████| 230M/230M [00:01<00:00, 162MB/s] 


epoch 0 train_loss:--------------------- 1.323
epoch 0 val loss------------------ 1.019


In [15]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
@author: sylvain
"""

# Calcul des statistiques sur un jeu de test


import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import torch
import torchvision.transforms as T
from torch.autograd import Variable
from skimage import io
import numpy as np
import pickle
import os

def do_confusion_matrix(all_mat, old_vocab, new_vocab, dataset):
    print(new_vocab)
    new_mat = np.zeros((len(new_vocab), len(new_vocab)))
    for i in range(1,all_mat.shape[0]):
        answer = old_vocab[i]
        new_i = new_vocab.index(answer)
        for j in range(1,all_mat.shape[1]):
            answer = old_vocab[j]
            new_j = new_vocab.index(answer)
            new_mat[new_i, new_j] = all_mat[i, j]

    if len(old_vocab) > 20:#HR
        new_mat = new_mat[0:18,0:18]
        new_vocab = new_vocab[0:18]
    fig = plt.figure(figsize=(10,5))
    ax = fig.add_subplot(111)
    cax = ax.matshow(np.log(new_mat+1), cmap="YlGn")
    fig.colorbar(cax)
    ax.set_xticklabels([''] + new_vocab)
    ax.set_yticklabels([''] + new_vocab)
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    fig.savefig('confusion_matrix_' + dataset + '.svg')

def get_vocab(dataset):
    data_path = '/kaggle/input/vqars1/6344334/'
    if dataset == "LR":
        allanswersJSON = os.path.join(data_path, 'all_answers.json')
        encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = True)
    else:
        allanswersJSON = os.path.join(data_path, 'USGSanswers.json')
        encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = False)
        
    return encoder_answers.getVocab()

def run(experiment, dataset, shuffle=False, num_batches=-1, save_output=False):
    print ('---' + experiment + '---')
    batch_size = 100
    data_path = '/kaggle/input/vqars1/6344334/'
    if dataset == "LR":
        allquestionsJSON = os.path.join(data_path, 'all_questions.json')
        allanswersJSON = os.path.join(data_path, 'all_answers.json')
        questionsJSON = os.path.join(data_path, 'LR_split_test_questions.json')
        answersJSON = os.path.join(data_path, 'LR_split_test_answers.json')
        imagesJSON = os.path.join(data_path, 'LR_split_test_images.json')
        images_path = os.path.join(data_path, 'Images_LR/')
        encoder_questions = VocabEncoder(allquestionsJSON, questions=True)
        encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = True)
        patch_size = 256
        categories = ['rural_urban', 'presence', 'count', 'comp']
    else:
        allquestionsJSON = os.path.join(data_path, 'USGSquestions.json')
        allanswersJSON = os.path.join(data_path, 'USGSanswers.json')
        if dataset == "HR":
            questionsJSON = os.path.join(data_path, 'USGS_split_test_questions.json')
            answersJSON = os.path.join(data_path, 'USGS_split_test_answers.json')
            imagesJSON = os.path.join(data_path, 'USGS_split_test_images.json')
        else:
            questionsJSON = os.path.join(data_path, 'USGS_split_test_phili_questions.json')
            answersJSON = os.path.join(data_path, 'USGS_split_test_phili_answers.json')
            imagesJSON = os.path.join(data_path, 'USGS_split_test_phili_images.json')
        images_path = os.path.join(data_path, 'dataUSGS/')
        encoder_questions = VocabEncoder(allquestionsJSON, questions=True)
        encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = False)
        patch_size = 512
        categories = ['area', 'presence', 'count', 'comp']

    weight_file =  experiment + '.pth'
    network = VQAModel(encoder_questions.getVocab(), encoder_answers.getVocab(), input_size = patch_size).cuda()
    state = network.state_dict()
    state.update(torch.load(weight_file))
    network.load_state_dict(state)
    network.eval().cuda()
    
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]
    transform = T.Compose([
        T.ToTensor(),            
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
      ])
    test_dataset = VQALoader(images_path, imagesJSON, questionsJSON, answersJSON, encoder_questions, encoder_answers, train=False, ratio_images_to_use=1, transform=transform, patch_size = patch_size)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

    countQuestionType = {cat: 0 for cat in categories}
    rightAnswerByQuestionType = {cat: 0 for cat in categories}
    true_positives = {cat: 0 for cat in categories}
    false_positives = {cat: 0 for cat in categories}
    false_negatives = {cat: 0 for cat in categories}
    confusionMatrix = np.zeros((len(encoder_answers.getVocab()), len(encoder_answers.getVocab())))
    
    for i, data in enumerate(test_loader, 0):
        if num_batches == 0:
            break
        num_batches -= 1
        if i % 100 == 99:
            print(float(i) / len(test_loader))
        question, answer, image, type_str, image_original = data
        question_str = [encoder_questions.decode(q.cpu().numpy()) for q in question]
        answer = Variable(answer.long()).cuda().resize_(len(question_str))
        if shuffle:
            order = np.array(range(image.shape[0]))
            np.random.shuffle(order)
            image[np.array(range(image.shape[0]))] = image[order]
        image = Variable(image.float()).cuda()
        pred = network(image, question_str)
        
        answer = answer.cpu().numpy()
        pred = np.argmax(pred.cpu().detach().numpy(), axis=1)
        
        for j in range(len(question_str)):
            category = type_str[j]
            countQuestionType[category] += 1
            if answer[j] == pred[j]:
                rightAnswerByQuestionType[category] += 1
                true_positives[category] += 1
            else:
                false_positives[category] += 1
                false_negatives[category] += 1
            confusionMatrix[answer[j], pred[j]] += 1
            
        if save_output:
            out_path = experiment + '_' + 'output' + dataset
            if not os.path.exists(out_path):
                os.mkdir(out_path)
            for j in range(len(question_str)):
                viz_img = T.ToPILImage()(image_original[j].float().data.cpu())
                viz_question = question_str[j]
                viz_answer = encoder_answers.decode([answer[j]])
                viz_pred = encoder_answers.decode([pred[j]])
            
                imname = str(i * len(question_str) + j) + '_q_' + viz_question + '_gt_' + viz_answer + '_pred_' + viz_pred + '.png'
                plt.imsave(os.path.join(out_path, imname), viz_img)
    
    Accuracies = {'AA': 0}
    for category in categories:
        Accuracies[category] = rightAnswerByQuestionType[category] * 1.0 / countQuestionType[category]
        Accuracies['AA'] += Accuracies[category] / len(categories)
    Accuracies['OA'] = np.trace(confusionMatrix)/np.sum(confusionMatrix)
    
    print('- Accuracies')
    for category in categories:
        print(f' - {category}: {Accuracies[category]:.4f}')
    print(f'- AA: {Accuracies["AA"]:.4f}')
    print(f'- OA: {Accuracies["OA"]:.4f}')
    
    # Compute precision, recall, and F1-score
    metrics = {}
    for category in categories:
        tp = true_positives[category]
        fp = false_positives[category]
        fn = false_negatives[category]
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        metrics[category] = {
            'precision': precision,
            'recall': recall,
            'f1': f1
        }

    print('- Precision, Recall, and F1-score')
    for category in categories:
        print(f' - {category}:')
        print(f'   Precision: {metrics[category]["precision"]:.4f}')
        print(f'   Recall: {metrics[category]["recall"]:.4f}')
        print(f'   F1-score: {metrics[category]["f1"]:.4f}')

    return Accuracies, confusionMatrix, metrics

expes = {'LR': ['/kaggle/working/ModelRSVQA']}
run('/kaggle/working/ModelRSVQA', 'LR', num_batches=5, save_output=True)

for dataset in expes.keys():
    acc = []
    mat = []
    all_metrics = []
    for experiment_name in expes[dataset]:
        if not os.path.isfile(experiment_name + 'accuracies_' + '.npy'):
            if dataset[-1] == 's':
                print("run", dataset[:-1])
                tmp_acc, tmp_mat, tmp_metrics = run(experiment_name, dataset[:-1], shuffle=True)
            else:
                dataset = 'LR'
                tmp_acc, tmp_mat, tmp_metrics = run(experiment_name, dataset)
            np.save(experiment_name + 'accuracies_', tmp_acc)
            np.save(experiment_name + 'confusion_matrix_', tmp_mat)
            np.save(experiment_name + 'metrics_', tmp_metrics)
        else:
            tmp_acc = np.load(experiment_name + 'accuracies_' + '.npy', allow_pickle=True)[()]
            tmp_mat = np.load(experiment_name + 'confusion_matrix_'+ '.npy', allow_pickle=True)[()]
            tmp_metrics = np.load(experiment_name + 'metrics_'+ '.npy', allow_pickle=True)[()]

        acc.append(tmp_acc)
        mat.append(tmp_mat)
        all_metrics.append(tmp_metrics)
        
    print('--- Total (' + dataset + ') ---')
    print('- Accuracies')
    for type_str in tmp_acc.keys():
        all_acc = []
        for tmp_acc in acc:
            all_acc.append(tmp_acc[type_str])
        print(f' - {type_str}: {np.mean(all_acc):.4f} (stddev = {np.std(all_acc):.4f})')
    
    if dataset[-1] == 's':
        vocab = get_vocab(dataset[:-1])
    else:
        vocab = get_vocab(dataset)

    all_mat = np.zeros(tmp_mat.shape)    
    for tmp_mat in mat:
        all_mat += tmp_mat
    
    if dataset[0] == 'H':
        new_vocab = ['yes', 'no', '0m2', 'between 0m2 and 10m2', 'between 10m2 and 100m2', 'between 100m2 and 1000m2', 'more than 1000m2'] + [str(i) for i in range(90)]
    else:
        new_vocab = ['yes', 'no', 'rural', 'urban', '0', 'between 0 and 10', 'between 10 and 100', 'between 100 and 1000', 'more than 1000']
        
    do_confusion_matrix(all_mat, vocab, new_vocab, dataset)

    print('- Precision, Recall, and F1-score')
    for category in tmp_metrics.keys():
        precision = np.mean([m[category]['precision'] for m in all_metrics])
        recall = np.mean([m[category]['recall'] for m in all_metrics])
        f1 = np.mean([m[category]['f1'] for m in all_metrics])
        print(f' - {category}:')
        print(f'   Precision: {precision:.4f}')
        print(f'   Recall: {recall:.4f}')
        print(f'   F1-score: {f1:.4f}')

---/kaggle/working/ModelRSVQA---
- Accuracies
 - rural_urban: 0.0000
 - presence: 0.6065
 - count: 0.2387
 - comp: 0.4278
- AA: 0.3182
- OA: 0.4220
- Precision, Recall, and F1-score
 - rural_urban:
   Precision: 0.0000
   Recall: 0.0000
   F1-score: 0.0000
 - presence:
   Precision: 0.6065
   Recall: 0.6065
   F1-score: 0.6065
 - count:
   Precision: 0.2387
   Recall: 0.2387
   F1-score: 0.2387
 - comp:
   Precision: 0.4278
   Recall: 0.4278
   F1-score: 0.4278
---/kaggle/working/ModelRSVQA---
0.9801980198019802
- Accuracies
 - rural_urban: 0.0000
 - presence: 0.6054
 - count: 0.2908
 - comp: 0.4290
- AA: 0.3313
- OA: 0.4361
- Precision, Recall, and F1-score
 - rural_urban:
   Precision: 0.0000
   Recall: 0.0000
   F1-score: 0.0000
 - presence:
   Precision: 0.6054
   Recall: 0.6054
   F1-score: 0.6054
 - count:
   Precision: 0.2908
   Recall: 0.2908
   F1-score: 0.2908
 - comp:
   Precision: 0.4290
   Recall: 0.4290
   F1-score: 0.4290
--- Total (LR) ---
- Accuracies
 - AA: 0.3313 (st

/tmp/ipykernel_34/2266345737.py:39: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_xticklabels([''] + new_vocab)
/tmp/ipykernel_34/2266345737.py:40: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax.set_yticklabels([''] + new_vocab)


- Precision, Recall, and F1-score
 - rural_urban:
   Precision: 0.0000
   Recall: 0.0000
   F1-score: 0.0000
 - presence:
   Precision: 0.6054
   Recall: 0.6054
   F1-score: 0.6054
 - count:
   Precision: 0.2908
   Recall: 0.2908
   F1-score: 0.2908
 - comp:
   Precision: 0.4290
   Recall: 0.4290
   F1-score: 0.4290


In [16]:
# #!/usr/bin/env python3
# # -*- coding: utf-8 -*-
# """
# @author: sylvain
# """

# # Calcul des statistiques sur un jeu de test

# #import VocabEncoder
# #import VQALoader
# #from models import model
# import matplotlib.pyplot as plt
# import matplotlib.ticker as ticker

# import torch
# import torchvision.transforms as T
# from torch.autograd import Variable
# from skimage import io
# import numpy as np
# import pickle
# import os

# def do_confusion_matrix(all_mat, old_vocab, new_vocab, dataset):
#     print(new_vocab)
#     new_mat = np.zeros((len(new_vocab), len(new_vocab)))
#     for i in range(1,all_mat.shape[0]):
#         answer = old_vocab[i]
#         new_i = new_vocab.index(answer)
#         for j in range(1,all_mat.shape[1]):
#             answer = old_vocab[j]
#             new_j = new_vocab.index(answer)
#             new_mat[new_i, new_j] = all_mat[i, j]

#     if len(old_vocab) > 20:#HR
#         new_mat = new_mat[0:18,0:18]
#         new_vocab = new_vocab[0:18]
#     fig = plt.figure(figsize=(10,5))
#     ax = fig.add_subplot(111)
#     cax = ax.matshow(np.log(new_mat+1), cmap="YlGn")
#     #plt.title('Confusion matrix of the classifier')
#     fig.colorbar(cax)
#     ax.set_xticklabels([''] + new_vocab)
#     ax.set_yticklabels([''] + new_vocab)
#     ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
#     ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
#     plt.xlabel('Predicted')
#     plt.ylabel('True')
#     plt.show()
#     fig.savefig('confusion_matrix_' + dataset + '.svg')
#     #plt.close()

        

# def get_vocab(dataset):
#     data_path = '/kaggle/input/vqars1/6344334/'
#     if dataset == "LR":
#         allanswersJSON = os.path.join(data_path, 'all_answers.json')
#         encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = True)
#     else:
#         allanswersJSON = os.path.join(data_path, 'USGSanswers.json')
#         encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = False)
        
#     return encoder_answers.getVocab()

# def run(experiment, dataset, shuffle=False, num_batches=-1, save_output=False):
#     print ('---' + experiment + '---')
#     batch_size = 100
#     data_path = '/kaggle/input/vqars1/6344334/'
#     if dataset == "LR":
#         allquestionsJSON = os.path.join(data_path, 'all_questions.json')
#         allanswersJSON = os.path.join(data_path, 'all_answers.json')
#         questionsJSON = os.path.join(data_path, 'LR_split_test_questions.json')
#         answersJSON = os.path.join(data_path, 'LR_split_test_answers.json')
#         imagesJSON = os.path.join(data_path, 'LR_split_test_images.json')
#         images_path = os.path.join(data_path, 'Images_LR/')
#         encoder_questions = VocabEncoder(allquestionsJSON, questions=True)
#         encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = True)
#         patch_size = 256
#     else:
#         allquestionsJSON = os.path.join(data_path, 'USGSquestions.json')
#         allanswersJSON = os.path.join(data_path, 'USGSanswers.json')
#         if dataset == "HR":
#             questionsJSON = os.path.join(data_path, 'USGS_split_test_questions.json')
#             answersJSON = os.path.join(data_path, 'USGS_split_test_answers.json')
#             imagesJSON = os.path.join(data_path, 'USGS_split_test_images.json')
#         else:
#             questionsJSON = os.path.join(data_path, 'USGS_split_test_phili_questions.json')
#             answersJSON = os.path.join(data_path, 'USGS_split_test_phili_answers.json')
#             imagesJSON = os.path.join(data_path, 'USGS_split_test_phili_images.json')
#         images_path = os.path.join(data_path, 'dataUSGS/')
#         encoder_questions = VocabEncoder(allquestionsJSON, questions=True)
#         encoder_answers = VocabEncoder(allanswersJSON, questions=False, range_numbers = False)
#         patch_size = 512

#     weight_file =  experiment + '.pth'
#     network = VQAModel(encoder_questions.getVocab(), encoder_answers.getVocab(), input_size = patch_size).cuda()
#     state = network.state_dict()
#     state.update(torch.load(weight_file))
#     network.load_state_dict(state)
#     network.eval().cuda()
    
#     IMAGENET_MEAN = [0.485, 0.456, 0.406]
#     IMAGENET_STD = [0.229, 0.224, 0.225]
#     transform = T.Compose([
#         T.ToTensor(),            
#         T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
#       ])
#     test_dataset = VQALoader(images_path, imagesJSON, questionsJSON, answersJSON, encoder_questions, encoder_answers, train=False, ratio_images_to_use=1, transform=transform, patch_size = patch_size)
#     test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

#     if dataset == 'LR':
#         countQuestionType = {'rural_urban': 0, 'presence': 0, 'count': 0, 'comp': 0}
#         rightAnswerByQuestionType = {'rural_urban': 0, 'presence': 0, 'count': 0, 'comp': 0}
#     else:
#         countQuestionType = {'area': 0, 'presence': 0, 'count': 0, 'comp': 0}
#         rightAnswerByQuestionType = {'area': 0, 'presence': 0, 'count': 0, 'comp': 0}
#     confusionMatrix = np.zeros((len(encoder_answers.getVocab()), len(encoder_answers.getVocab())))
    
#     for i, data in enumerate(test_loader, 0):
#         if num_batches == 0:
#             break
#         num_batches -= 1
#         if i % 100 == 99:
#             print(float(i) / len(test_loader))
#         question, answer, image, type_str, image_original = data
#         question = Variable(question.long()).cuda()
#         answer = Variable(answer.long()).cuda().resize_(question.shape[0])
#         if shuffle:
#             order = np.array(range(image.shape[0]))
#             np.random.shuffle(order)
#             image[np.array(range(image.shape[0]))] = image[order]
#         image = Variable(image.float()).cuda()
#         pred = network(image,question)
        
#         answer = answer.cpu().numpy()
#         pred = np.argmax(pred.cpu().detach().numpy(), axis=1)
        
#         for j in range(answer.shape[0]):
#             countQuestionType[type_str[j]] += 1
#             if answer[j] == pred[j]:
#                 rightAnswerByQuestionType[type_str[j]] += 1
#             confusionMatrix[answer[j], pred[j]] += 1
            
#         if save_output:
#             out_path = experiment + '_' + 'output' + dataset
#             if not os.path.exists(out_path):
#                 os.mkdir(out_path)
#             for j in range(batch_size):
#                 viz_img = T.ToPILImage()(image_original[j].float().data.cpu())
#                 viz_question = encoder_questions.decode(question[j].data.cpu().numpy())
#                 viz_answer = encoder_answers.decode([answer[j]])
#                 viz_pred = encoder_answers.decode([pred[j]])
            
#                 imname = str(i * batch_size + j) + '_q_' + viz_question + '_gt_' + viz_answer + '_pred_' + viz_pred + '.png'
#                 plt.imsave(os.path.join(out_path, imname), viz_img)
    
#     Accuracies = {'AA': 0}
#     for type_str in countQuestionType.keys():
#         Accuracies[type_str] = rightAnswerByQuestionType[type_str] * 1.0 / countQuestionType[type_str]
#         Accuracies['AA'] += Accuracies[type_str] / len(countQuestionType.keys())
#     Accuracies['OA'] = np.trace(confusionMatrix)/np.sum(confusionMatrix)
    
#     print('- Accuracies')
#     for type_str in countQuestionType.keys():
#         print (' - ' + type_str + ': ' + str(Accuracies[type_str]))
#     print('- AA: ' + str(Accuracies['AA']))
#     print('- OA: ' + str(Accuracies['OA']))
    
#     return Accuracies, confusionMatrix


# #expes = {'LRs': ['427f37d306ef4d03bb1406d5cd20336f', 'bd1387960b624257b9a50924d8134be6', '899e11235c624ec9bbb66e26da52d6fc'],
# #         'LR': ['427f37d306ef4d03bb1406d5cd20336f', 'bd1387960b624257b9a50924d8134be6', '899e11235c624ec9bbb66e26da52d6fc'],
# #         'HR': ['65f94a4f7ccd491da362f73e46795d26', '988853ae5d5e441695f98ee506021bdf', '3bfd251cafb74d379d02bf59d383381a'],
# #         'HRPhili': ['65f94a4f7ccd491da362f73e46795d26', '988853ae5d5e441695f98ee506021bdf', '3bfd251cafb74d379d02bf59d383381a'],
# #         'HRs': ['65f94a4f7ccd491da362f73e46795d26', '988853ae5d5e441695f98ee506021bdf', '3bfd251cafb74d379d02bf59d383381a'],
# #         'HRPhilis': ['65f94a4f7ccd491da362f73e46795d26', '988853ae5d5e441695f98ee506021bdf', '3bfd251cafb74d379d02bf59d383381a']}
# expes = {'LR': ['/kaggle/working/ModelRSVQA']}
# #run('65f94a4f7ccd491da362f73e46795d26', 'HR', num_batches=5, save_output=True)
# #run('65f94a4f7ccd491da362f73e46795d26', 'HRPhili', num_batches=5, save_output=True)
# run('/kaggle/working/ModelRSVQA', 'LR', num_batches=5, save_output=True)
# for dataset in expes.keys():
#     acc = []
#     mat = []
#     for experiment_name in expes[dataset]:
#         if not os.path.isfile(experiment_name + 'accuracies_' + '.npy'):
#             if dataset[-1] == 's':
#                 print("run", dataset[:-1])
#                 tmp_acc, tmp_mat = run(experiment_name, dataset[:-1], shuffle=True)
#             else:
#                 dataset = 'LR'
#                 tmp_acc, tmp_mat = run(experiment_name, dataset)
#             np.save(experiment_name + 'accuracies_', tmp_acc)
#             np.save(experiment_name + 'confusion_matrix_', tmp_mat)
#         else:
#             tmp_acc = np.load(experiment_name + 'accuracies_' + '.npy', allow_pickle=True)[()]
#             tmp_mat = np.load(experiment_name + 'confusion_matrix_'+ '.npy', allow_pickle=True)[()]

#         acc.append(tmp_acc)
#         mat.append(tmp_mat)
        
#     print('--- Total (' + dataset + ') ---')
#     print('- Accuracies')
#     for type_str in tmp_acc.keys():
#         all_acc = []
#         for tmp_acc in acc:
#             all_acc.append(tmp_acc[type_str])
#         print(' - ' + type_str + ': ' + str(np.mean(all_acc)) + ' ( stddev = ' + str(np.std(all_acc)) + ')')
    
#     if dataset[-1] == 's':
#         vocab = get_vocab(dataset[:-1])
#     else:
#         vocab = get_vocab(dataset)

#     all_mat = np.zeros(tmp_mat.shape)    
#     for tmp_mat in mat:
#         all_mat += tmp_mat
    
#     if dataset[0] == 'H':
#         new_vocab = ['yes', 'no', '0m2', 'between 0m2 and 10m2', 'between 10m2 and 100m2', 'between 100m2 and 1000m2', 'more than 1000m2'] + [str(i) for i in range(90)]
#     else:
#         new_vocab = ['yes', 'no', 'rural', 'urban', '0', 'between 0 and 10', 'between 10 and 100', 'between 100 and 1000', 'more than 1000']
        
#     do_confusion_matrix(all_mat, vocab, new_vocab, dataset)


# #labels = ['Yes', 'No', '<=10', '0', '<=100', '<=1000', '>1000', 'Rural', 'Urban']
# #fig = plt.figure()
# #ax = fig.add_subplot(111)
# #cax = ax.matshow(np.log(confusionMatrix[1:,1:] + 1), cmap="YlGn")
# ##plt.title('Confusion matrix of the classifier')
# #fig.colorbar(cax)
# #ax.set_xticklabels([''] + labels)
# #ax.set_yticklabels([''] + labels)
# #plt.xlabel('Predicted')
# #plt.ylabel('True')
# #plt.show()
# #fig.savefig(os.path.join(baseFolder, 'AccMatrix.pdf'))
# #print(Accuracies)

In [17]:
# import numpy as np

# # Load the confusion matrix from .npy file
# confusion_matrix = np.load('/kaggle/working/ModelRSVQAconfusion_matrix_.npy')


In [18]:
# import matplotlib.pyplot as plt

# # Display the confusion matrix
# plt.imshow(confusion_matrix, interpolation='nearest', cmap=plt.cm.Blues)
# plt.title('Confusion Matrix')
# plt.colorbar()

# # Add labels
# plt.xlabel('Predicted labels')
# plt.ylabel('True labels')

# # Show plot
# plt.show()


In [19]:
''''import matplotlib.pyplot as plt
import torch
import torchvision.transforms as T
from PIL import Image

def visualize_samples(model, validate_dataset, encoder_answers, num_samples=5):
    model.eval()
    dataloader = torch.utils.data.DataLoader(validate_dataset, batch_size=1, shuffle=True)
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(15, 5*num_samples))
    
    with torch.no_grad():
        for i, (question, answer, image, _, image_original) in enumerate(dataloader):
            if i >= num_samples:
                break
            
            question_str = validate_dataset.encoder_questions.decode(question[0].cpu().numpy())
            image = image.float().cuda()
            
            # Get model prediction
            pred = model(image, [question_str])
            pred_idx = pred.argmax(dim=1).item()
            
            # Decode ground truth and prediction
            ground_truth = encoder_answers.decode([answer[0].item()])
            prediction = encoder_answers.decode([pred_idx])
            
            # Display image
            axes[i, 0].imshow(image_original[0].permute(1, 2, 0))
            axes[i, 0].axis('off')
            axes[i, 0].set_title(f"Question: {question_str}")
            
            # Display ground truth and prediction
            axes[i, 1].axis('off')
            axes[i, 1].text(0.1, 0.6, f"Ground Truth: {ground_truth}", fontsize=12)
            axes[i, 1].text(0.1, 0.4, f"Prediction: {prediction}", fontsize=12)
            
            color = 'green' if ground_truth == prediction else 'red'
            axes[i, 1].text(0.1, 0.2, "Correct" if ground_truth == prediction else "Incorrect", 
                            fontsize=14, color=color, weight='bold')
    
    plt.tight_layout()
    plt.savefig('sample_predictions.png')
    plt.show()

# After training your model
#model.eval()

# Visualize sample predictions
visualize_samples("/kaggle/input/bertattentionnewbeg/BertAttentionRemoteMyHR16.pth", validate_dataset, encoder_answers)'''

'\'import matplotlib.pyplot as plt\nimport torch\nimport torchvision.transforms as T\nfrom PIL import Image\n\ndef visualize_samples(model, validate_dataset, encoder_answers, num_samples=5):\n    model.eval()\n    dataloader = torch.utils.data.DataLoader(validate_dataset, batch_size=1, shuffle=True)\n    \n    fig, axes = plt.subplots(num_samples, 2, figsize=(15, 5*num_samples))\n    \n    with torch.no_grad():\n        for i, (question, answer, image, _, image_original) in enumerate(dataloader):\n            if i >= num_samples:\n                break\n            \n            question_str = validate_dataset.encoder_questions.decode(question[0].cpu().numpy())\n            image = image.float().cuda()\n            \n            # Get model prediction\n            pred = model(image, [question_str])\n            pred_idx = pred.argmax(dim=1).item()\n            \n            # Decode ground truth and prediction\n            ground_truth = encoder_answers.decode([answer[0].item()])\n   

In [20]:
# print(confusion_matrix)

In [21]:
# # Get non-zero indices
# nonzero_indices = np.nonzero(confusion_matrix)

# # Get non-zero values
# nonzero_values = confusion_matrix[nonzero_indices]

# # Plot non-zero values
# plt.figure(figsize=(20, 20))
# plt.scatter(nonzero_indices[1], nonzero_indices[0], s=nonzero_values*10, c=nonzero_values, cmap='Blues', edgecolor='black')
# plt.colorbar(label='Count')
# plt.xlabel('Predicted labels')
# plt.ylabel('True labels')
# plt.title('Confusion Matrix')
# plt.grid(True)
# plt.show()

In [22]:
# import numpy as np

# # Load the confusion matrix from .npy file
# confusion_matrix = np.load('/kaggle/working/ModelRSVQAconfusion_matrix_.npy')

# # Get non-zero indices and values
# nonzero_indices = np.nonzero(confusion_matrix)
# nonzero_values = confusion_matrix[nonzero_indices]

# # Print non-zero values and their indices
# for i in range(len(nonzero_values)):
#     idx = (nonzero_indices[0][i], nonzero_indices[1][i])
#     val = nonzero_values[i]
#     print(f"Non-zero value: {val} at index {idx}")


In [23]:
# import seaborn as sns

# plt.figure(figsize=(8, 6))
# sns.heatmap(confusion_matrix, annot=True, fmt='.2f', cmap='Blues') 

# plt.xlabel('Predicted Label')
# plt.ylabel('True Label')
# plt.show()


In [24]:
# import numpy as np
# import seaborn as sns
# import matplotlib.pyplot as plt

# # Load the confusion matrix from .npy file
# confusion_matrix = np.load('/kaggle/working/ModelRSVQAconfusion_matrix_.npy')

# # Plot the confusion matrix using Seaborn
# plt.figure(figsize=(8, 6))
# sns.heatmap(confusion_matrix, annot=True, cmap='Blues', fmt='g')
# plt.title('Confusion Matrix')
# plt.xlabel('Predicted labels')
# plt.ylabel('True labels')
# plt.show()
